# 04 — Clean Holdings: Dedupe + Asset Classification

**The "Apple duplicate" fix:** one security appears in *multiple* infoTable rows when voting authority / investment discretion is split across subsidiaries. Those rows are the **same position** — we aggregate on `(quarter, cusip, put_call)`, summing value/shares and keeping an `n_rows` audit column.

`put_call` stays in the key so a stock and a put on the same CUSIP remain **separate positions** (they are economically opposite).

Then every position gets an `asset_class` (Common Stock, Preferred, ADR/ADS, ETF, Corporate/Convertible Bond, Note, Call/Put Option, Right, Warrant) and a `portfolio_weight`.

In [ ]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

In [ ]:
from src.assets import clean_holdings

holdings = clean_holdings()
holdings.head(10)

In [ ]:
# Proof the dedupe worked: positions merged from >1 raw row
holdings[holdings["n_rows"] > 1][["quarter", "issuer", "n_rows", "shares", "value_usd"]].head(10)

In [ ]:
# Weights must sum to 1 within each quarter
holdings.groupby("quarter")["portfolio_weight"].sum().round(6)

In [ ]:
# Asset class mix, latest quarter
latest = holdings["quarter"].max()
holdings[holdings["quarter"] == latest].groupby("asset_class")["value_usd"].sum().sort_values(ascending=False)